# Run All

Runs the whole pipeline in order, from the raw CSV files to the deep-dive analysis:

| Step | Notebook |
| --- | --- |
| 1 | `01_data_setup.ipynb` |
| 2 | `02_data_quality.ipynb` |
| 3 | `03_data_cleaning.ipynb` |
| 4 | `04_delay_analysis.ipynb` |
| 5 | `05_deep_dive_analysis.ipynb` |

Each notebook is executed in a fresh kernel, with the `notebooks/` folder as its working directory, and saved in place with its outputs. The runner works whether it is started from the project root or from `notebooks/`. The run stops at the first notebook that fails: that notebook is marked `FAILED`, the ones after it `SKIPPED`, and this notebook ends with an error.

## 1. Setup

In [4]:
import time
import traceback
from pathlib import Path

import nbformat
import pandas as pd
from nbconvert.preprocessors import ExecutePreprocessor

PIPELINE = [
    "01_data_setup.ipynb",
    "02_data_quality.ipynb",
    "03_data_cleaning.ipynb",
    "04_delay_analysis.ipynb",
    "05_deep_dive_analysis.ipynb",
]
TIMEOUT_SEC = 1200

# find notebooks/ from the working directory: works from the project root, from notebooks/ or from a subfolder
candidates = [d / sub for d in [Path.cwd(), *Path.cwd().parents] for sub in ("", "notebooks")]
NOTEBOOK_DIR = next((d.resolve() for d in candidates if (d / "00_run_all.ipynb").is_file()), None)
assert NOTEBOOK_DIR, f"Cannot find the notebooks folder from {Path.cwd()}"
PROJECT_ROOT = NOTEBOOK_DIR.parent

missing = [name for name in PIPELINE if not (NOTEBOOK_DIR / name).is_file()]
assert not missing, f"Missing notebooks: {missing}"
print(f"Working directory: {Path.cwd()}")
print(f"Notebooks:         {NOTEBOOK_DIR}")

Working directory: e:\Important\Skills\railpulse-railway-analytics\notebooks
Notebooks:         E:\Important\Skills\railpulse-railway-analytics\notebooks


## 2. Run the notebooks

Strictly in the order of `PIPELINE`. Each notebook runs with `notebooks/` as its working directory, the same as when it is opened by hand. Any error, including a timeout or a crashed kernel, stops the run.

In [5]:
results = []
failure = None

for name in PIPELINE:
    if failure:
        results.append((name, "SKIPPED", 0.0, "earlier notebook failed"))
        continue

    path = NOTEBOOK_DIR / name
    nb = nbformat.read(path, as_version=4)
    executor = ExecutePreprocessor(timeout=TIMEOUT_SEC, kernel_name="python3")

    start = time.time()
    try:
        executor.preprocess(nb, {"metadata": {"path": str(NOTEBOOK_DIR)}})
        status, detail = "OK", ""
    except Exception as err:  # cell errors, timeouts and dead kernels all stop the pipeline
        status = "FAILED"
        detail = f"{type(err).__name__}: {str(err).strip().splitlines()[-1] if str(err).strip() else ''}"
        failure = (name, err)
    finally:
        # keep the outputs, including those of a failed run, for debugging
        nbformat.write(nb, path)

    results.append((name, status, round(time.time() - start, 1), detail))
    print(f"{status:7} {name} ({results[-1][2]:.1f}s)")

summary = pd.DataFrame(results, columns=["notebook", "status", "seconds", "detail"])
print()
print(summary.to_string(index=False))

if failure:
    name, err = failure
    print(f"\nPIPELINE FAILED at {name}. Open that notebook to see the failing cell.")
    traceback.print_exception(err)
    raise RuntimeError(f"Pipeline failed at {name}") from err
print("\nPIPELINE OK: all notebooks ran in order")

OK      01_data_setup.ipynb (1.6s)
OK      02_data_quality.ipynb (2.0s)
OK      03_data_cleaning.ipynb (1.9s)
OK      04_delay_analysis.ipynb (2.4s)
OK      05_deep_dive_analysis.ipynb (15.2s)

                   notebook status  seconds detail
        01_data_setup.ipynb     OK      1.6       
      02_data_quality.ipynb     OK      2.0       
     03_data_cleaning.ipynb     OK      1.9       
    04_delay_analysis.ipynb     OK      2.4       
05_deep_dive_analysis.ipynb     OK     15.2       

PIPELINE OK: all notebooks ran in order


## 3. Summary

In [6]:
summary

,notebook,status,seconds,detail
0,01_data_setup.ipynb,OK,1.6,
1,02_data_quality.ipynb,OK,2.0,
2,03_data_cleaning.ipynb,OK,1.9,
3,04_delay_analysis.ipynb,OK,2.4,
4,05_deep_dive_analysis.ipynb,OK,15.2,
